# 训推引擎组 Harness · 能力包在 Colab 里试跑

跑两个能力包（`inference-perf` 推理性能 / `cache-loss` 缓存损耗）的 reference validator，验证：L0 预检 + L1 验收能输出 `HARNESS_METRIC`、主指标方向（越大/越小）择优、以及 GPU 吞吐微基准。

> 怎么看输出：**绿勾 = 没报错**；真正的输出在每个 cell **正下方**。关键数字在第 3 步和第 6 步 cell 下方。

## 1. 环境（有没有 GPU）

In [ ]:
import shutil, sys
print("Python:", sys.version.split()[0])
print("nvidia-smi:", "yes" if shutil.which("nvidia-smi") else "no")
try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch 未安装（GPU 微基准会跳过）:", e)


## 2. 落地两个 validate.py（base64 内嵌，避免任何转义问题）

In [ ]:
import base64, pathlib
INF_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiLmjqjnkIbmgKfog73og73lipvljIUgcmVmZXJlbmNlIHZhbGlkYXRvcu+8iOacgOWwj+Wunui3te+8ieOAggoK55So6YCU77ya6K+B5piO44CMTDAg6aKE5qOAIOKGkiBMMSDpqozmlLYg4oaSIEhBUk5FU1NfTUVUUklDIOKGkiDkuLvmjIfmoIcgdHBzKG1heGltaXplKSDihpIg5pyA5LyY6L2u5qyh44CN6L+Z5p2h6ZO+6Lev5Y+v6LeR6YCa44CCCgrnlKjms5XvvJoKICAgIHB5dGhvbjMgdmFsaWRhdGUucHkgLS1wcmVjaGVjayAgICAgICAgIyBMMO+8muivreazlS/lpZHnuqboh6rmo4DvvIzlv6vpgJ/nn63ot6/vvIjkuI3ot5Hln7rlh4bvvIkKICAgIHB5dGhvbjMgdmFsaWRhdGUucHkgLS12YWxpZGF0ZSAgICAgICAgIyBMMe+8mui3keS4gOS4quWwjyBDUFUg5Z+65YeG77yM6L6T5Ye6IEhBUk5FU1NfTUVUUklDCgrnnJ/lrp7ku7vliqHvvJrmioogcnVuX2JlbmNobWFyayDmjaLmiJAgc2dsYW5nL2JlbmNobWFya19zZXJ2aW5nIOetieecn+WunuWOi+a1i++8jArlubbmioogSEFSTkVTU19NRVRSSUMg55qEIHRwcy90dGZ0X21zIOaNouaIkOecn+WunumHh+mbhuWAvO+8m+acrOaWh+S7tuWPquS/neivgeWlkee6puato+ehruOAgeWPr+aXoCBHUFUg5aSN546w44CCCiIiIgppbXBvcnQganNvbgppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgoKZGVmIF93b3JrKGl0ZXJhdGlvbnM6IGludCkgLT4gZmxvYXQ6CiAgICAiIiLkuIDkuKrlj6/ooqvmqKHlnovpgJrov4fmlLnku6PnoIEv566X5rOV5p2l5LyY5YyW55qE5Y+v6K6h566X6LSf6L2977yI5ZCe5ZCQ55qE5Luj55CG5Z+65YeG77yJ44CCIiIiCiAgICBhY2MgPSAwLjAKICAgIGZvciBpIGluIHJhbmdlKGl0ZXJhdGlvbnMpOgogICAgICAgIGFjYyArPSAoaSAqIDI2NTQ0MzU3NjEpICUgMTAwMAogICAgcmV0dXJuIGFjYwoKCmRlZiBydW5fYmVuY2htYXJrKGl0ZXJhdGlvbnM6IGludCk6CiAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIF93b3JrKGl0ZXJhdGlvbnMpCiAgICBkdCA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MAogICAgdHBzID0gaXRlcmF0aW9ucyAvIGR0IGlmIGR0ID4gMCBlbHNlIDAuMCAgICAgICMg5q+P56eS6L+t5Luj5pWw77yI5ZCe5ZCQ5Luj55CG5oyH5qCH77yJCiAgICB0dGZ0X21zID0gZHQgKiAxMDAwLjAgICAgICAgICAgICAgICAgICAgICAgICAgICMg56uv5Yiw56uv6ICX5pe277yI5Luj55CGIFRURlTvvIzop4Llr5/pobnvvIkKICAgIHJldHVybiB0cHMsIHR0ZnRfbXMKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIGFyZ3MgPSBzeXMuYXJndlsxOl0KICAgIGl0ZXJhdGlvbnMgPSAzXzAwMF8wMDAKICAgIGlmICItLWl0ZXJhdGlvbnMiIGluIGFyZ3M6CiAgICAgICAgaSA9IGFyZ3MuaW5kZXgoIi0taXRlcmF0aW9ucyIpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpdGVyYXRpb25zID0gbWF4KDEsIGludChhcmdzW2kgKyAxXSkpCiAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBJbmRleEVycm9yKToKICAgICAgICAgICAgcGFzcwoKICAgIGlmICItLXByZWNoZWNrIiBpbiBhcmdzOgogICAgICAgIF93b3JrKDEwKQogICAgICAgIHByaW50KCJQUkVDSEVDS19PSzog6K+t5rOVL+Wlkee6puiHquajgOmAmui/hyIpCiAgICAgICAgcmV0dXJuIDAKCiAgICB0cHMsIHR0ZnRfbXMgPSBydW5fYmVuY2htYXJrKGl0ZXJhdGlvbnMpCiAgICBwcmludChqc29uLmR1bXBzKHsidHBzIjogcm91bmQodHBzLCAxKSwgInR0ZnRfbXMiOiByb3VuZCh0dGZ0X21zLCAyKX0pKQogICAgIyDmmL7lvI8gSEFSTkVTU19NRVRSSUMg5aWR57qm77yaTG9vcCDku47ov5nph4zlj5bkuLvmjIfmoIfvvIhKU09OIOW9ouW8j++8ieOAggogICAgcHJpbnQoZidIQVJORVNTX01FVFJJQzoge3sidHBzIjoge3JvdW5kKHRwcywgMSl9LCAidHRmdF9tcyI6IHtyb3VuZCh0dGZ0X21zLCAyKX19fScpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg=="
CACHE_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiLnvJPlrZjmjZ/ogJfog73lipvljIUgcmVmZXJlbmNlIHZhbGlkYXRvcu+8iOacgOWwj+Wunui3te+8jOesrOS6jOaWueWQke+8mui2iuWwj+i2iuWlve+8ieOAggoK55So6YCU77ya6K+B5piO44CMSEFSTkVTU19NRVRSSUMgbG9zc19wY3QobWluaW1pemUp44CN6L+Z5p2h44CM56ys5LqM5pa55ZCR44CN6ZO+6Lev5Y+v6LeR6YCa4oCU4oCUCuS4jiBpbmZlcmVuY2UtcGVyZiDnmoQgbWF4aW1pemUg5pa55ZCR55u45a+577yM6K+05piOIExvb3Ag55qE5pyA5LyY6L2u5qyh5q+U6L6D5LiN57uR5a6a5Y2V5LiA5oyH5qCH5pa55ZCR44CCCgrnlKjms5XvvJoKICAgIHB5dGhvbjMgdmFsaWRhdGUucHkgLS1wcmVjaGVjayAgICAgICAgIyBMMO+8muivreazlS/lpZHnuqboh6rmo4AKICAgIHB5dGhvbjMgdmFsaWRhdGUucHkgLS12YWxpZGF0ZSAgICAgICAgIyBMMe+8mui3keS4gOS4quehruWumuaAp+e8k+WtmOaooeaLn++8jOi+k+WHuiBIQVJORVNTX01FVFJJQ++8iEtWIOW9ouW8j++8iQoK55yf5a6e5Lu75Yqh77ya5o2i5oiQIEF0dGVudGlvblN0b3JlL0FTQ2FjaGUg55qE5ZG95Lit546H5LiO5o2f6ICX6YeH6ZuG77yb6L+Z6YeM55So56Gu5a6a5oCn5qih5ouf5L+d6K+B5Y+v5pegIEdQVSDlpI3njrDjgIIKIiIiCmltcG9ydCByYW5kb20KaW1wb3J0IHN5cwoKCmRlZiBydW5fc2ltKGFjY2Vzc2VzOiBpbnQgPSAyMDAwLCBjYXBhY2l0eTogaW50ID0gNjQsIHNlZWQ6IGludCA9IDcpOgogICAgcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkKQogICAgY2FjaGUgPSB7fQogICAgaGl0cyA9IDAKICAgIGZvciBfIGluIHJhbmdlKGFjY2Vzc2VzKToKICAgICAgICBrZXkgPSBybmcucmFuZGludCgwLCBjYXBhY2l0eSAqIDQpCiAgICAgICAgaWYga2V5IGluIGNhY2hlOgogICAgICAgICAgICBoaXRzICs9IDEKICAgICAgICBlbHNlOgogICAgICAgICAgICBjYWNoZVtrZXldID0gVHJ1ZQogICAgICAgICAgICBpZiBsZW4oY2FjaGUpID4gY2FwYWNpdHk6CiAgICAgICAgICAgICAgICBjYWNoZS5wb3AobmV4dChpdGVyKGNhY2hlKSkpICAjIOeugOWNlemaj+acuua3mOaxsAogICAgbG9zc19wY3QgPSByb3VuZCgoMSAtIGhpdHMgLyBhY2Nlc3NlcykgKiAxMDAsIDIpCiAgICBjYWNoZV9oaXQgPSByb3VuZChoaXRzIC8gYWNjZXNzZXMsIDQpCiAgICByZXR1cm4gbG9zc19wY3QsIGNhY2hlX2hpdAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgYXJncyA9IHN5cy5hcmd2WzE6XQogICAgaWYgIi0tcHJlY2hlY2siIGluIGFyZ3M6CiAgICAgICAgcHJpbnQoIlBSRUNIRUNLX09LOiDor63ms5Uv5aWR57qm6Ieq5qOA6YCa6L+HIikKICAgICAgICByZXR1cm4gMAoKICAgIGxvc3NfcGN0LCBjYWNoZV9oaXQgPSBydW5fc2ltKCkKICAgICMg5pi+5byPIEhBUk5FU1NfTUVUUklDIOWlkee6pu+8iEtWIOW9ouW8j++8jOS4u+aMh+aghyBsb3NzX3BjdO+8ieOAggogICAgcHJpbnQoZiJIQVJORVNTX01FVFJJQzogbG9zc19wY3Q9e2xvc3NfcGN0fSBjYWNoZV9oaXQ9e2NhY2hlX2hpdH0iKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgc3lzLmV4aXQobWFpbigpKQo="
inf = pathlib.Path("inference-perf/validate.py"); inf.parent.mkdir(parents=True, exist_ok=True)
cache = pathlib.Path("cache-loss/validate.py"); cache.parent.mkdir(parents=True, exist_ok=True)
inf.write_text(base64.b64decode(INF_B64).decode("utf-8"))
cache.write_text(base64.b64decode(CACHE_B64).decode("utf-8"))
for p in (inf, cache):
    data = p.read_text()
    print(f"{p}: {len(data)} bytes | 首行: {data.splitlines()[0]}")
    assert len(data) > 50, f"{p} 写入异常（只有 {len(data)} 字节）"
print("落地完成 ✓（文件非空、内容就位）")


## 3. 跑两个能力包的 L0 预检 + L1 验收（进程内直跑，不经过 subprocess）

In [ ]:
import sys
def run_validator(script, flag):
    sys.argv = [script, flag]
    code = open(script, encoding="utf-8").read()
    try:
        exec(compile(code, script, "exec"), {"__name__": "__main__", "__file__": script})
    except SystemExit:
        pass  # 脚本末尾 sys.exit(main()) 正常结束

for name, script in [("inference-perf", "inference-perf/validate.py"), ("cache-loss", "cache-loss/validate.py")]:
    print("=" * 60)
    print(name)
    for flag in ("--precheck", "--validate"):
        print("$ python3", script, flag)
        run_validator(script, flag)
    print()


## 4. （可选）GPU 真实吞吐微基准：PyTorch matmul 的 tps 代理

In [ ]:
import json, time
try:
    import torch
    has = True
except Exception:
    has = False

if has:
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    n, iters = 4096, 20
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    for _ in range(3):
        torch.matmul(a, b)
    if dev == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        torch.matmul(a, b)
    if dev == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    matmuls_per_sec = iters / dt
    tflops = matmuls_per_sec * 2 * n**3 / 1e12
    print(f"device={dev} matmuls/sec={matmuls_per_sec:.1f} TFLOPS={tflops:.2f}")
    print(f'HARNESS_METRIC: {{"tps": {round(matmuls_per_sec, 1)}, "tflops": {round(tflops, 2)}}}')
else:
    print("torch 未安装，跳过 GPU 微基准（第 3 步的 CPU 版已经给出 tps）。")


## 5. 主指标 + 方向：最优轮次怎么选（与 metrics.ts 同逻辑）

In [ ]:
def metric_score(metric, direction):
    if metric is None:
        return None
    return metric if direction == "maximize" else -metric

def pick_better(baseline, candidate, direction):
    bs, cs = metric_score(baseline, direction), metric_score(candidate, direction)
    if cs is None:
        return "baseline (candidate 无指标)"
    if bs is None or cs > bs:
        return "candidate"
    return "baseline"

print("inference-perf (maximize tps):",
      "基线 100 -> 优化 210 胜者:", pick_better(100, 210, "maximize"),
      "| 基线 210 -> 退化 100 胜者:", pick_better(210, 100, "maximize"))
print("cache-loss    (minimize loss_pct):",
      "基线 74.6 -> 优化 30.1 胜者:", pick_better(74.6, 30.1, "minimize"),
      "| 基线 30.1 -> 退化 74.6 胜者:", pick_better(30.1, 74.6, "minimize"))


## 6. 汇总：两个能力包的主指标（只看这一格也行）

In [ ]:
import contextlib, io, sys
def metric_line(script):
    sys.argv = [script, "--validate"]
    code = open(script, encoding="utf-8").read()
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        try:
            exec(compile(code, script, "exec"), {"__name__": "__main__", "__file__": script})
        except SystemExit:
            pass
    for l in buf.getvalue().splitlines():
        if l.startswith("HARNESS_METRIC:"):
            return l
    return "(未找到 HARNESS_METRIC —— 说明第 2 步「落地」没成功)"

print("=" * 66)
print("能力包                         主指标（HARNESS_METRIC）")
print("=" * 66)
for name, script, direction in [
    ("inference-perf（tps，越大越好）", "inference-perf/validate.py", "maximize"),
    ("cache-loss（loss_pct，越小越好）", "cache-loss/validate.py", "minimize"),
]:
    print(f"{name:34s} -> {metric_line(script)}   （方向={direction}）")
print("=" * 66)
